In [177]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import random
import math
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [178]:
ratings_file_path = "archive/ratings_small.csv"

df_ratings = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "sopanmagar/moviesmetadata",
  ratings_file_path,
)

Using Colab cache for faster access to the 'moviesmetadata' dataset.


In [179]:
file_path = "archive/movies_metadata.csv"

df_full = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "sopanmagar/moviesmetadata",
  file_path,
)

Using Colab cache for faster access to the 'moviesmetadata' dataset.


/usr/local/lib/python3.12/dist-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [180]:
df_full['id'] = pd.to_numeric(df_full['id'], errors='coerce')
df_full.dropna(subset=['id'], inplace=True)
df_full.drop_duplicates(subset=['id'], inplace=True)
df_full['id'] = df_full['id'].astype(int)

movie_id_to_title_map = dict(zip(df_full['id'], df_full['title']))
df_ratings['title'] = df_ratings['movieId'].map(movie_id_to_title_map)
df_ratings.dropna(subset=['title'], inplace=True)

df = df_full[df_full['id'].isin(df_ratings['movieId'].unique())].copy()
df.drop_duplicates(subset='title', inplace=True)
df.reset_index(drop=True, inplace=True)

In [181]:
m = df['vote_count'].quantile(0.90)
C = df['vote_average'].mean()

def weighted_rating(row, m=m, C=C):
    v = row['vote_count']
    R = row['vote_average']
    return round((v / (v + m)) * R + (m / (v + m)) * C, 2)

df['score'] = df.apply(weighted_rating, axis=1)

cols_to_numeric = ['budget', 'popularity']

for col in cols_to_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['budget_log'] = np.log(df['budget'])
df['popularity_log'] = np.log(df['popularity'])

df.describe()

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


,budget,id,popularity,revenue,runtime,vote_average,vote_count,score,budget_log,popularity_log
count,2.794000e+03,2794.000000,2794.000000,2.794000e+03,2791.000000,2794.000000,2794.000000,2794.000000,2794.000000,2794.000000
mean,1.389620e+07,17211.869005,6.307884,4.783980e+07,105.356503,6.316034,429.384037,6.402788,-inf,-inf
std,3.105097e+07,28396.171483,7.020868,1.245048e+08,27.385200,1.327958,1002.922917,0.266528,NaN,NaN
min,0.000000e+00,2.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,5.150000,-inf,-inf
25%,0.000000e+00,1371.250000,1.392271,0.000000e+00,92.000000,5.800000,11.000000,6.310000,NaN,0.330936
50%,0.000000e+00,3175.500000,4.942156,0.000000e+00,103.000000,6.500000,54.000000,6.320000,NaN,1.597801
75%,1.300000e+07,26166.250000,9.671148,3.200000e+07,118.000000,7.100000,346.750000,6.390000,16.380460,2.269147
max,3.800000e+08,160718.000000,140.950236,1.845034e+09,320.000000,10.000000,12269.000000,8.230000,19.755682,4.948407


In [182]:
df = df.drop(columns=['adult', 'belongs_to_collection','homepage','imdb_id','original_title','poster_path','production_countries', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'video', 'vote_count', 'vote_average', 'budget'], errors='ignore')

In [183]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['overview'].fillna(''))

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim

indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def get_recommendations(title, cosine_sim=cosine_sim, df=df, indices=indices):
    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:6]

    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]


In [184]:
print("Рекомендации для фильма:")
display(get_recommendations('Heat'))

Рекомендации для фильма:


,title
1290,Masques
1557,Kiss Kiss Bang Bang
2001,Van Gogh
1316,Ocean's Twelve
1279,Catwoman


In [185]:
user_movie_matrix = df_ratings.pivot(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

In [186]:
def recommend(user_id, n=5):
    movie_titles = dict(zip(df_full['id'], df_full['title']))

    if user_id not in user_movie_matrix.index:
        popular = df_ratings.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(n)
        return pd.Series([movie_titles.get(mid, f'Movie {mid}') for mid in popular.index])

    user_sim = cosine_similarity(user_movie_matrix)
    user_sim_df = pd.DataFrame(user_sim, index=user_movie_matrix.index, columns=user_movie_matrix.index)
    similar_users = user_sim_df[user_id].sort_values(ascending=False)[1:21]
    watched = df_ratings[df_ratings['userId'] == user_id]['movieId'].values

    scores = {}
    for uid, sim in similar_users.items():
        for _, row in df_ratings[df_ratings['userId'] == uid].iterrows():
            if row['movieId'] not in watched:
                scores[row['movieId']] = scores.get(row['movieId'], []) + [row['rating'] * sim]

    avg_scores = {mid: np.mean(s) for mid, s in scores.items()}
    top = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)[:n]

    return pd.Series([movie_titles.get(mid, f'Movie {mid}') for mid, score in top])

In [187]:
print("Рекомендации для пользователя 1:")
print(recommend(1, 5))

print("\nРекомендации для нового пользователя:")
print(recommend(9999, 5))

Рекомендации для пользователя 1:
0                        My Own Private Idaho
1                           American Graffiti
2                       Breaking and Entering
3            The Garden of the Finzi-Continis
4    Harry Potter and the Philosopher's Stone
dtype: object

Рекомендации для нового пользователя:
0          Design of Death
1    Psychopathia Sexualis
2             Emma's Bliss
3                Innocence
4      Night Without Sleep
dtype: object


In [188]:
#Коробочная версия
from sklearn.decomposition import NMF

In [189]:
user_movie_matrix = df_ratings.pivot(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

In [190]:
nmf = NMF(n_components=20, init='random', random_state=42)
user_features = nmf.fit_transform(user_movie_matrix)
movie_features = nmf.components_.T

predicted_ratings = np.dot(user_features, movie_features.T)
predicted_df = pd.DataFrame(
    predicted_ratings,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.columns
)

def recommend_nmf(user_id, n=5):
    movie_titles = dict(zip(df_full['id'], df_full['title']))

    if user_id not in predicted_df.index:
        popular = df_ratings.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(n)
        return pd.Series([movie_titles.get(mid, f'Movie {mid}') for mid in popular.index])

    user_predictions = predicted_df.loc[user_id]
    watched = df_ratings[df_ratings['userId'] == user_id]['movieId'].values
    unwatched = user_predictions[~user_predictions.index.isin(watched)]
    top_movies = unwatched.sort_values(ascending=False).head(n)

    return pd.Series([movie_titles.get(mid, f'Movie {mid}') for mid in top_movies.index])

/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [191]:
print("Рекомендации NMF для пользователя 1:")
print(recommend_nmf(1, 5))

print("\nРекомендации NMF для нового пользователя:")
print(recommend_nmf(9999, 5))

Рекомендации NMF для пользователя 1:
0    The Man with the Golden Arm
1                       Rocky IV
2                   The 39 Steps
3                  Sweet Sixteen
4                     Princesses
dtype: object

Рекомендации NMF для нового пользователя:
0          Design of Death
1    Psychopathia Sexualis
2             Emma's Bliss
3                Innocence
4      Night Without Sleep
dtype: object


In [192]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['overview'].fillna(''))

nmf_movies = NMF(n_components=20, init='random', random_state=42)
movie_features_nmf = nmf_movies.fit_transform(tfidf_matrix)
topic_features = nmf_movies.components_.T

reconstructed_matrix = np.dot(movie_features_nmf, topic_features.T)
reconstructed_df = pd.DataFrame(
    reconstructed_matrix,
    index=df['title'],
    columns=tfidf_vectorizer.get_feature_names_out()
)

In [193]:
def recommend_by_description(title, n=5):
    if title not in reconstructed_df.index:
        return pd.Series([])

    movie_vector = reconstructed_df.loc[title]

    similarities = {}
    for other_title in reconstructed_df.index:
        if other_title != title:
            other_vector = reconstructed_df.loc[other_title]
            sim = cosine_similarity([movie_vector], [other_vector])[0][0]
            similarities[other_title] = sim

    top_similar = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:n]

    return pd.Series([title for title, sim in top_similar])

In [194]:
print("Рекомендации:")
print(recommend_by_description('Heat', 5))

Рекомендации:
0      The Squeeze
1    Union Station
2        Blackmail
3           Frenzy
4      Angel Heart
dtype: object


In [195]:
movie_title = 'Heat'
user_id = 1

recommendations_tfidf = get_recommendations(movie_title, cosine_sim, df, indices)
recommendations_nmf_content = recommend_by_description(movie_title)
recommendations_user_based_cf = recommend(user_id)
recommendations_nmf_cf = recommend_nmf(user_id)

comparison_df = pd.DataFrame({
    'Контент': recommendations_tfidf.reset_index(drop=True),
    'Контент NMF': recommendations_nmf_content.reset_index(drop=True),
    'Юзеры': recommendations_user_based_cf.reset_index(drop=True),
    'Юзеры NMF': recommendations_nmf_cf.reset_index(drop=True)
})

display(comparison_df)

,Контент,Контент NMF,Юзеры,Юзеры NMF
0,Masques,The Squeeze,My Own Private Idaho,The Man with the Golden Arm
1,Kiss Kiss Bang Bang,Union Station,American Graffiti,Rocky IV
2,Van Gogh,Blackmail,Breaking and Entering,The 39 Steps
3,Ocean's Twelve,Frenzy,The Garden of the Finzi-Continis,Sweet Sixteen
4,Catwoman,Angel Heart,Harry Potter and the Philosopher's Stone,Princesses
